# ACER — EU AI Act Real-Requirement Experiment

This notebook uses the source-traceable EU AI Act requirement corpus and the architecture-aware scenario generator. It is the research analysis layer.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from app.engine import assess
from app.loaders import load_requirements, load_system
from app.adaptation import load_tactics
from app.agents import ComplianceOrchestrator
reqs=load_requirements(ROOT/'data/requirements/eu_ai_act_core.yaml')
tactics=load_tactics(ROOT/'data/tactics/eu_ai_act_tactics.yaml')

In [ ]:
from scripts.generate_ai_act_scenarios import write_scenarios
generated=ROOT/'data/systems/eu_ai_act_scenarios'
write_scenarios(generated,n=30,seed=42)

In [ ]:
rows=[]
for p in sorted(generated.glob('ai-act-scenario-*.yaml')):
    s=load_system(p)
    b=assess(s,reqs)
    f,h=ComplianceOrchestrator(reqs,tactics).run(s)
    a=assess(f,reqs)
    rows.append({'system_id':s.id,'before':b.overall_status,'after':a.overall_status,
                 'before_violations':sum(x.status=='FAIL' for x in b.results),
                 'after_violations':sum(x.status=='FAIL' for x in a.results),
                 'steps':sum(x.get('accepted',False) for x in h)})
scenario_df=pd.DataFrame(rows)
scenario_df.head()

In [ ]:
print('Final compliant:',(scenario_df.after=='COMPLIANT').sum(),'/',len(scenario_df))
print('Success rate:',round(100*(scenario_df.after=='COMPLIANT').mean(),2),'%')
print('Mean accepted steps:',round(scenario_df.steps.mean(),2))

In [ ]:
ax=scenario_df[['before_violations','after_violations']].mean().plot(kind='bar',figsize=(7,4))
ax.set_title('Mean EU AI Act requirement violations before vs after adaptation')
ax.set_ylabel('Average failed requirements')
plt.tight_layout(); plt.show()

## Research caveat

This remains a controlled benchmark until the ground-truth labels are independently validated. The notebook should later compare manual, LLM-assisted, MBSE, and agentic pipelines using the same scenarios.